In [39]:
from transformers import pipeline
import re
from rapidfuzz import fuzz

In [40]:


def normalize_text(text: str) -> str:
    text = text.lower()

    replacements = {
        '3': 'e', '4': 'a', '5': 's',
        '0': 'o', '@': 'a', '$': 's'
    }
    for k, v in replacements.items():
        text = text.replace(k, v)

    text = text.replace('1', 'l')

    text = re.sub(r'[^a-z0-9\s]', '', text)

    text = re.sub(
        r'(\b[a-z]\s+){2,}[a-z]\b',
        lambda m: m.group(0).replace(" ", ""),
        text
    )

    text = re.sub(r'\s+', ' ', text).strip()

    return text


def fuzzy_check(text):
    # targets = ["kill", "suicide", "rape", "nude"]
    targets = []

    for word in targets:
        score = max(
            fuzz.partial_ratio(text, word),
            fuzz.ratio(text, word)
        )

        if score > 80:
            return True, word

    return False, None

In [41]:
from transformers import pipeline

model_1 = pipeline(
    "text-classification",
    model="unitary/unbiased-toxic-roberta",
    return_all_scores=True
)

def extract_scores(output):
    """
    Handles both:
    - single label output
    - multi-label output
    """

    # Case 1: list of dicts (correct multi-label)
    if isinstance(output, list) and isinstance(output[0], dict):
        return {r["label"].lower(): r["score"] for r in output}

    # Case 2: nested list
    if isinstance(output, list) and isinstance(output[0], list):
        return {r["label"].lower(): r["score"] for r in output[0]}

    # Case 3: single dict
    if isinstance(output, dict):
        return {output["label"].lower(): output["score"]}

    return {}

def predict_model_1(text):
    output = model_1(text)[0]

    label = output["label"]
    score = output["score"]

    # IMPORTANT: LABEL_1 = NON-toxic
    if label == "LABEL_1":
        return 1 - score   # invert
    else:
        return score

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19161.65it/s]
RobertaForSequenceClassification LOAD REPORT from: unitary/unbiased-toxic-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
def decide(score):
    if score > 0.9:
        return "BLOCKED"
    elif score > 0.6:
        return "FLAGGED"
    else:
        return "APPROVED"

In [43]:
def moderate_with_model(text, predict_fn):
    processed = normalize_text(text)

    # Fuzzy check first
    fuzzy_hit, word = fuzzy_check(processed)
    if fuzzy_hit:
        return {
            "text": text,
            "processed": processed,
            "status": "BLOCKED",
            "reason": "fuzzy_match",
            "word": word
        }

    # Model prediction
    score = predict_fn(processed)

    return {
        "text": text,
        "processed": processed,
        "toxicity": score,
        "status": decide(score)
    }

In [44]:
tests = [
    "Hello, how are you?",
    "I hate you",
    "You are stupid",
    "Send me nudes",
    "I will kill you",
    "k i l l",
    "k!ll",
    "ki11",
    "Let's build something amazing",
    "You are annoying",
    "Go to hell",
    "kill time"
]
print("Model 1 output :-")
for t in tests:
    print("TEXT:", t)
    print("Model 1:", moderate_with_model(t, predict_model_1))
    print("-" * 80)

Model 1 output :-
TEXT: Hello, how are you?
Model 1: {'text': 'Hello, how are you?', 'processed': 'hello how are you', 'toxicity': 0.007151651196181774, 'status': 'APPROVED'}
--------------------------------------------------------------------------------
TEXT: I hate you
Model 1: {'text': 'I hate you', 'processed': 'i hate you', 'toxicity': 0.8707150220870972, 'status': 'FLAGGED'}
--------------------------------------------------------------------------------
TEXT: You are stupid
Model 1: {'text': 'You are stupid', 'processed': 'you are stupid', 'toxicity': 0.9969064593315125, 'status': 'BLOCKED'}
--------------------------------------------------------------------------------
TEXT: Send me nudes
Model 1: {'text': 'Send me nudes', 'processed': 'send me nudes', 'toxicity': 0.03361578285694122, 'status': 'APPROVED'}
--------------------------------------------------------------------------------
TEXT: I will kill you
Model 1: {'text': 'I will kill you', 'processed': 'i will kill you', 